<div dir="rtl">

# گزارش تمرین اول مبانی بینایی کامپیوتر

این گزارش نتایج پردازش تصویر فلفل، بازیابی تصویر سند و بازسازی ویدئوی پارک را با تکیه بر سلول‌های اجرایی و خروجی‌های ثبت‌شده ارائه می‌کند. تمام مقادیر عددی به پارامترهای همین آزمایش و فایل‌های ورودی موجود مربوط‌اند.

</div>

<div dir="rtl">

## بخش اول: تحلیل و جداسازی رنگ‌ها در تصاویر دیجیتال

</div>

<div dir="rtl">

### ۱. بررسی مشخصات تصویر

**صورت مسئله و خواسته تمرین:** تصویر `Peppers.png` بارگذاری و ابعاد، نوع داده، تعداد و ترتیب کانال‌ها و محدودهٔ مقادیر پیکسل گزارش شود.

**روش کار و منطق ریاضی:** تابع `imread` با حالت رنگی تصویری به شکل `(height, width, channels)` تولید می‌کند. کمینه و بیشینه روی تمام عناصر آرایه محاسبه می‌شوند. برای نمایش در Matplotlib، ترتیب BGR به RGB تبدیل شده است؛ اندازه‌گیری‌ها بر آرایهٔ اصلی انجام می‌شوند.

**شواهد عددی و نتایج تجربی:**

| مشخصه | مقدار |
|---|---|
| ارتفاع × عرض | 1024 × 1536 پیکسل |
| شکل آرایه | `(1024, 1536, 3)` |
| نوع داده | `uint8` |
| تعداد و ترتیب کانال‌ها | 3، به ترتیب BGR |
| کمینه و بیشینهٔ مشاهده‌شده | 0 و 255 |

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### چرا ترتیب کانال‌ها در نمایش اهمیت دارد؟
OpenCV تصویر رنگی بارگذاری‌شده را با ترتیب آبی، سبز و قرمز نگه می‌دارد، در حالی که Matplotlib دادهٔ سه‌کاناله را RGB تفسیر می‌کند. تبدیل ترتیب از جابه‌جایی ظاهری قرمز و آبی جلوگیری می‌کند و تغییری در دادهٔ مبنای محاسبات این قسمت ایجاد نمی‌کند.

</div>

In [ ]:
from pathlib import Path
from typing import Tuple, Union

import cv2
import matplotlib.pyplot as plt

image_path = Path("Media_src/Peppers.png")
image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

if image is None:
    raise FileNotFoundError(f"Could not load image: {image_path}")

In [ ]:
height, width, channels = image.shape

print(f"Image dimensions (height x width): {height} x {width} pixels")
print(f"Array shape (height, width, channels): {image.shape}")
print(f"Data type: {image.dtype}")
print(f"Number of color channels: {channels}")
print("Channel order: BGR (Blue, Green, Red)")
print(f"Pixel value range: {image.min()} to {image.max()}")

In [ ]:
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(7, 5))
plt.imshow(image_rgb)
# plt.imshow(image)
plt.title("Peppers")
plt.axis("off")
plt.show()

<div dir="rtl">

### ۲. تبدیل انواع تصاویر

**صورت مسئله و خواسته تمرین:** تصویر ورودی در حالت‌های RGB، خاکستری و دودویی نمایش داده شود؛ روش دودویی‌سازی و آستانه مشخص و اطلاعات ازدست‌رفته تحلیل شوند.

**روش کار و منطق ریاضی:** تصویر خاکستری با تبدیل استاندارد OpenCV به دست می‌آید که تقریباً از رابطهٔ $Y=0.299R+0.587G+0.114B$ استفاده می‌کند. آستانه‌گذاری سراسری (Global Thresholding) با $T=127$ به صورت $B(x,y)=255$ اگر $Y(x,y)>127$ و در غیر این صورت $B(x,y)=0$ اجرا می‌شود. تصویر RGB با تغییر ترتیب کانال‌های تصویر ورودی ایجاد شده است.

**شواهد عددی و نتایج تجربی:** خروجی دارای سه نمایش RGB، خاکستری و دودویی است؛ آستانهٔ ثابت برابر 127 و دو مقدار مجاز تصویر دودویی 0 و 255 هستند.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### در تبدیل رنگی به خاکستری چه اطلاعاتی حذف می‌شود؟
سه مؤلفهٔ رنگی در یک شدت خلاصه می‌شوند؛ ازاین‌رو رنگ‌فام (Hue) و اشباع (Saturation) به صورت مستقل قابل بازیابی نیستند. دو سطح با رنگ متفاوت ممکن است روشنایی خاکستری مشابه داشته باشند و تفکیک آن‌ها دشوار شود.

#### آستانه‌گذاری چه اطلاعات بیشتری را حذف می‌کند؟
شدت‌های پیوسته در دو گروه قرار می‌گیرند؛ جزئیات بافت و اختلاف‌های روشنایی درون هر گروه از بین می‌روند. آستانهٔ سراسری به سایه و نورپردازی حساس است و تضمین نمی‌کند که اجسام با روشنایی متفاوت به‌درستی از پس‌زمینه جدا شوند.

</div>

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
threshold = 127
_, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, picture, title in zip(
    axes,
    [image_rgb, gray, binary],
    ["RGB", "Grayscale", "Binary (threshold = 127)"],
):
    ax.imshow(
        picture, cmap="gray" if picture.ndim == 2 else None, vmin=0, vmax=255
    )
    # ax.imshow(picture, vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۳. بررسی فضاهای رنگی RGB و HSV

**صورت مسئله و خواسته تمرین:** تصویر در دو فضای RGB و HSV تجزیه شود؛ کانال‌ها جداگانه نمایش داده و مناسب‌بودن هر فضا برای تفکیک رنگ تحلیل شود.

**روش کار و منطق ریاضی:** در RGB شدت‌های $R,G,B$ مستقیماً نمایش داده می‌شوند. تبدیل HSV مؤلفه‌های رنگ‌فام $H$، اشباع $S$ و روشنایی $V$ را استخراج می‌کند؛ در نمایش ۸ بیتی OpenCV بازهٔ $H$ برابر 0 تا 179 و بازهٔ $S,V$ برابر 0 تا 255 است. هر کانال با مقیاس ثابت مربوط به خودش نمایش داده می‌شود.

**شواهد عددی و نتایج تجربی:** شش تصویر حاصل شامل سه کانال RGB و سه کانال HSV هستند. مقیاس ثابت $H$ تا 179 و سایر کانال‌ها تا 255 تنظیم شده است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### اطلاعات RGB و HSV چه تفاوتی دارند؟
در RGB، اثر رنگ و روشنایی در سه مؤلفه درهم‌آمیخته است؛ بنابراین تغییر نور می‌تواند هر سه مقدار را تغییر دهد. HSV رنگ‌فام را از شاخص شدت و اشباع جدا می‌کند و امکان اعمال شرط مشخص روی رنگ را فراهم می‌سازد.

#### کدام فضا برای جداسازی رنگ مناسب‌تر است؟
برای آستانه‌گذاری رنگی این تمرین، HSV انتخاب مناسب‌تری است، زیرا بازهٔ رنگ‌فام با قیود اشباع و روشنایی ترکیب می‌شود. با این حال، رنگ‌فام در اشباع بسیار پایین ناپایدار است و سایه، نور شدید و تبدیل گسسته نیز می‌توانند نتیجه را تغییر دهند؛ بنابراین HSV نسبت به تغییر نور کاملاً ناوردا نیست.

</div>

In [ ]:
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, (color_image, names) in enumerate(
    [
        (image_rgb, ["Red", "Green", "Blue"]),
        (hsv, ["Hue", "Saturation", "Value"]),
    ]
):
    for index, name in enumerate(names):
        axes[row, index].imshow(
            color_image[:, :, index],
            cmap="gray",
            vmin=0,
            vmax=179 if name == "Hue" else 255,
        )
        axes[row, index].set_title(name)
        axes[row, index].axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۴. جداسازی نواحی رنگی

**صورت مسئله و خواسته تمرین:** نواحی سبز شناسایی شوند و ماسک دودویی، تصویر رنگی ماسک‌شده و معیارهای انتخاب ارائه شوند؛ تعلق نواحی انتخاب‌شده به فلفل‌های سبز بررسی شود.

**روش کار و منطق ریاضی:** در فضای HSV، ماسک $M(x,y)=255$ زمانی برقرار است که $35\le H\le85$، $50\le S\le255$ و $40\le V\le255$؛ در غیر این صورت صفر است. تصویر خروجی با اعمال ماسک بر تصویر BGR و سپس تبدیل برای نمایش ساخته می‌شود. آستانه‌های اشباع و روشنایی انتخاب پیکسل‌های کم‌رنگ یا بسیار تاریک را محدود می‌کنند.

**شواهد عددی و نتایج تجربی:** پوشش ماسک برابر **11.59٪** کل پیکسل‌ها است. تصاویر ثبت‌شده، ماسک و نتیجهٔ اعمال آن را در کنار تصویر اصلی نشان می‌دهند.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### آیا همهٔ نواحی شناسایی‌شده متعلق به فلفل‌های سبز هستند؟
خیر. ساقه‌های سبز فلفل‌های قرمز، زرد و نارنجی نیز در بازهٔ انتخاب قرار می‌گیرند. برعکس، بخشی از سطح فلفل سبز در سایه یا بازتاب روشن به علت عبور از حدود اشباع یا روشنایی حذف می‌شود. طبقه‌بندی پیکسلی فاقد اطلاعات شکل و هویت شیء است؛ برای انتساب هر پیکسل به بدنهٔ فلفل، اطلاعات مکانی یا مدل شناسایی شیء لازم خواهد بود.

</div>

In [ ]:
import numpy as np

# [Hue, Saturation, Value]
green_lower = np.array([35, 50, 40], dtype=np.uint8)
green_upper = np.array([85, 255, 255], dtype=np.uint8)

# green_lower = np.array([35, 5, 0], dtype=np.uint8)
# green_upper = np.array([85, 255, 255], dtype=np.uint8)


def extract_green(bgr_image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    hsv_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv_image, green_lower, green_upper)
    result = cv2.bitwise_and(bgr_image, bgr_image, mask=mask)
    return mask, result


green_mask, green_image = extract_green(image)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, picture, title in zip(
    axes,
    [image_rgb, green_mask, cv2.cvtColor(green_image, cv2.COLOR_BGR2RGB)],
    ["Original", "Green mask", "Extracted green"],
):
    ax.imshow(
        picture, cmap="gray" if picture.ndim == 2 else None, vmin=0, vmax=255
    )
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Green mask coverage: {np.mean(green_mask > 0) * 100:.2f}%")

<div dir="rtl">

### ۵. بررسی هیستوگرام و کنتراست

**صورت مسئله و خواسته تمرین:** هیستوگرام خاکستری و کنتراست RMS تصویر محاسبه و دو نسخه با کنتراست بیشتر و کمتر از نظر بصری و کمی مقایسه شوند.

**روش کار و منطق ریاضی:** با $I_i$ در مقیاس 0 تا 255 و $u_i=I_i/255$، معیار $C_{RMS}=\sqrt{N^{-1}\sum_i(u_i-\bar u)^2}$ است. تغییر کنتراست حول میانگین شدت اصلی $\mu$ به صورت $I'=\operatorname{clip}(\mu+\alpha(I-\mu),0,255)$ انجام و سپس به `uint8` تبدیل می‌شود؛ $\alpha=1.5$ برای افزایش و $0.5$ برای کاهش است.

**شواهد عددی و نتایج تجربی:**

| تصویر | کنتراست RMS |
|---|---:|
| اصلی | 0.1906 |
| کنتراست بیشتر | 0.2769 |
| کنتراست کمتر | 0.0953 |

کمینه، بیشینه و میانگین شدت تصویر خاکستری اصلی به‌ترتیب 0، 251 و 130.29 هستند. سهم شدت‌های 0–63، 64–127، 128–191 و 192–255 به‌ترتیب 13.9٪، 25.6٪، 51.4٪ و 9.1٪ است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### هیستوگرام و کنتراست چگونه تغییر کرده‌اند؟
بیشترین سهم پیکسل‌ها در بازهٔ 128 تا 191 قرار دارد. ضریب بزرگ‌تر از یک فاصلهٔ شدت‌ها از میانگین را زیاد می‌کند، ولی برش (Clipping) در 0 و 255 افزایش کنتراست را از حالت خطی کامل خارج می‌کند و ممکن است جزئیات سایه یا روشنایی را حذف کند. ضریب 0.5 توزیع را فشرده و RMS را تقریباً نصف کرده است؛ کمی اختلاف ناشی از گسسته‌سازی است.

</div>

In [ ]:
def rms_contrast(gray_image: np.ndarray) -> np.float64:
    return np.std(gray_image.astype(np.float64) / 255.0)


def change_contrast(gray_image: np.ndarray, factor: float) -> np.ndarray:
    mean = gray_image.mean()
    adjusted = (gray_image.astype(np.float64) - mean) * factor + mean
    return np.clip(adjusted, 0, 255).astype(np.uint8)


contrast_images = [
    gray,
    change_contrast(gray, 1.5),
    change_contrast(gray, 0.5),
]
contrast_names = ["Original", "Higher contrast", "Lower contrast"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for index, (picture, name) in enumerate(zip(contrast_images, contrast_names)):
    axes[0, index].imshow(picture, cmap="gray", vmin=0, vmax=255)
    axes[0, index].set_title(f"{name} (RMS = {rms_contrast(picture):.4f})")
    axes[0, index].axis("off")
    axes[1, index].hist(picture.ravel(), bins=256, range=(0, 256))
    axes[1, index].set_xlim(0, 255)
    axes[1, index].set_xlabel("Intensity")
    axes[1, index].set_ylabel("Pixel count")
plt.tight_layout()
plt.show()
for name, picture in zip(contrast_names, contrast_images):
    print(f"{name}: RMS = {rms_contrast(picture):.4f}")
print(
    f"Original grayscale: min = {gray.min()}, max = {gray.max()}, "
    f"mean = {gray.mean():.2f}"
)

<div dir="rtl">

### ۶. بهبود تصویر در شرایط نوری نامناسب

**صورت مسئله و خواسته تمرین:** تصویر تاریک شبیه‌سازی و با همسان‌سازی و کشش هیستوگرام بهبود داده شود؛ تصاویر، هیستوگرام‌ها، روش انتخابی و تفاوت‌ها گزارش شوند.

**روش کار و منطق ریاضی:** در HSV، کانال $V$ در 0.3 ضرب و به عدد صحیح تبدیل می‌شود؛ تصویر تاریک مجدداً به HSV تبدیل می‌شود تا ورودی یکسانی برای هر دو روش حاصل شود. همسان‌سازی (Histogram Equalization) از تابع توزیع تجمعی شدت برای نگاشت غیرخطی استفاده می‌کند. کشش (Histogram Stretching) با $V'=255(V-V_{min})/(V_{max}-V_{min})$ انجام می‌شود و در صورت برابر بودن دو حد، کانال بدون تغییر می‌ماند. هر دو روش تنها $V$ را تغییر می‌دهند و سپس تصویر به BGR برمی‌گردد.

**شواهد عددی و نتایج تجربی:**

| تصویر | بازهٔ V | میانگین V |
|---|---:|---:|
| تاریک | 0–76 | 45.50 |
| همسان‌سازی | 0–255 | 130.49 |
| کشش | 0–255 | 152.15 |

هیستوگرام‌های خروجی مربوط به کانال $V$ هستند، نه تصویر خاکستری.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### چرا پردازش کانال روشنایی انتخاب شد؟
تغییر مستقل کانال‌های RGB می‌تواند نسبت آن‌ها و در نتیجه رنگ‌فام را تغییر دهد. پردازش $V$ همراه با نگه‌داشتن $H,S$ در فضای میانی، رنگ را تا حدی حفظ می‌کند؛ بااین‌حال تبدیل رفت‌وبرگشت و گرد کردن، حفظ دقیق رنگ را تضمین نمی‌کنند.

#### تفاوت عملکرد دو روش چیست؟
همسان‌سازی به شکل هیستوگرام وابسته و غیرخطی است و می‌تواند نواحی پرتراکم را بیش از سایر شدت‌ها گسترش دهد. کشش خطی روابط نسبی شدت‌ها را تا زمان گرد کردن حفظ می‌کند، اما به مقادیر حدی و نقاط پرت حساس است. هیچ‌یک اطلاعات ازدست‌رفته در کوانتیزه‌سازی تصویر تاریک را بازسازی نمی‌کند.

</div>

In [ ]:
dark_hsv = hsv.copy()
dark_hsv[:, :, 2] = (hsv[:, :, 2].astype(np.float32) * 0.3).astype(np.uint8)
dark_image = cv2.cvtColor(dark_hsv, cv2.COLOR_HSV2BGR)

# Start from the same dark image for both enhancement methods.
dark_hsv = cv2.cvtColor(dark_image, cv2.COLOR_BGR2HSV)
dark_value = dark_hsv[:, :, 2]

equalized_hsv = dark_hsv.copy()
equalized_hsv[:, :, 2] = cv2.equalizeHist(dark_value)
equalized_image = cv2.cvtColor(equalized_hsv, cv2.COLOR_HSV2BGR)

stretched_hsv = dark_hsv.copy()
value_min, value_max = int(dark_value.min()), int(dark_value.max())
if value_max > value_min:
    stretched = (
        (dark_value.astype(np.float32) - value_min)
        * 255
        / (value_max - value_min)
    )
    stretched_hsv[:, :, 2] = np.clip(stretched, 0, 255).astype(np.uint8)
stretched_image = cv2.cvtColor(stretched_hsv, cv2.COLOR_HSV2BGR)

enhanced_images = [dark_image, equalized_image, stretched_image]
enhanced_names = ["Dark", "Histogram equalization", "Histogram stretching"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for index, (picture, name) in enumerate(zip(enhanced_images, enhanced_names)):
    axes[0, index].imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
    axes[0, index].set_title(name)
    axes[0, index].axis("off")
    value = cv2.cvtColor(picture, cv2.COLOR_BGR2HSV)[:, :, 2]
    axes[1, index].hist(value.ravel(), bins=256, range=(0, 256))
    axes[1, index].set_xlim(0, 255)
    axes[1, index].set_xlabel("Value (V)")
    axes[1, index].set_ylabel("Pixel count")
    print(
        f"{name}: V range = {value.min()} to {value.max()}, "
        f"mean = {value.mean():.2f}"
    )
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۷. بررسی تأثیر بهبود تصویر بر جداسازی رنگ

**صورت مسئله و خواسته تمرین:** ماسک سبز با آستانه‌های یکسان روی تصویر اصلی، تاریک و دو تصویر بهبودیافته محاسبه و مقایسه شود.

**روش کار و منطق ریاضی:** حدود HSV قسمت ۴ بدون تغییر به کار رفته‌اند. پوشش برابر $100|M|/N$ و اشتراک بر اجتماع (IoU) با مرجع اولیه برابر $|M\cap M_0|/|M\cup M_0|$ است؛ در صورت تهی بودن اجتماع، کد مقدار 1 را برمی‌گرداند.

**شواهد عددی و نتایج تجربی:**

| تصویر | پوشش ماسک | IoU با ماسک اصلی |
|---|---:|---:|
| اصلی | 11.59٪ | 1.000 |
| تاریک | 1.98٪ | 0.164 |
| همسان‌سازی | 5.02٪ | 0.421 |
| کشش | 11.82٪ | 0.981 |

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### روشنایی و بهبود آن چه اثری بر ماسک دارند؟
کاهش $V$ پیکسل‌هایی را به زیر آستانهٔ 40 می‌برد؛ به همین علت ماسک تاریک به‌شدت کوچک می‌شود. دو روش بهبود بخشی از پیکسل‌ها را بازمی‌گردانند، اما نگاشت غیرخطی همسان‌سازی در این آزمایش باعث شباهت کمتری به ماسک اصلی نسبت به کشش شده است. شباهت چشمگیر کشش به ارتباط تقریبی خطی آن با تاریک‌سازی انتخاب‌شده وابسته است و نتیجه‌ای عمومی برای همهٔ تصاویر نیست.

#### آیا IoU بالا به معنی تشخیص دقیق فلفل است؟
خیر. مرجع $M_0$ خروجی همان آستانه‌گذاری اولیه است و حقیقت مبنا (Ground Truth) محسوب نمی‌شود؛ در نتیجه خطاهای اولیه، از جمله انتخاب ساقه‌ها، می‌توانند در مقایسه تکرار شوند.

</div>

In [ ]:
comparison_images = [image, dark_image, equalized_image, stretched_image]
comparison_names = ["Original", "Dark", "Equalized", "Stretched"]
comparison_masks = [extract_green(picture)[0] for picture in comparison_images]
reference = green_mask > 0

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, mask, name in zip(axes, comparison_masks, comparison_names):
    selected = mask > 0
    union = np.logical_or(reference, selected).sum()
    iou = np.logical_and(reference, selected).sum() / union if union else 1.0
    coverage = selected.mean() * 100
    ax.imshow(mask, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{name}\nCoverage: {coverage:.2f}% | IoU: {iou:.3f}")
    ax.axis("off")
    print(f"{name}: coverage = {coverage:.2f}%, IoU with original = {iou:.3f}")
plt.tight_layout()
plt.show()

<div dir="rtl">

## بخش دوم: پردازش و بازیابی تصویر سند اسکن‌شده

</div>

<div dir="rtl">

### ۱. بارگذاری تصویر و اضافه کردن نویز

**صورت مسئله و خواسته تمرین:** مشخصات سند بررسی، نویز فلفل‌نمکی و گاوسی به تصویر افزوده و تصاویر و پارامترها مقایسه شوند.

**روش کار و منطق ریاضی:** تصویر رنگی به خاکستری تبدیل می‌شود. در نویز فلفل‌نمکی (Salt & Pepper)، هر موقعیت با احتمال 0.025 صفر و با احتمال 0.025 برابر 255 می‌شود؛ انتخاب مستقل است، پس تعداد واقعی نقاط تصادفی است. در نویز گاوسی، $I'=\operatorname{clip}(I+n,0,255)$، با $n\sim\mathcal N(0,25^2)$، سپس تبدیل به `uint8` انجام می‌شود. بذر مولد تصادفی 42 است.

**شواهد عددی و نتایج تجربی:** تصویر ورودی `(2048, 2048, 3)`، نوع `uint8` و ترتیب BGR دارد؛ تصویر خاکستری `(2048, 2048)` و بازهٔ واقعی آن 30 تا 248 است. چگالی مورد انتظار نویز ضربه‌ای 5٪ و انحراف معیار نویز گاوسی 25 است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### ماهیت دو نویز چه تفاوتی دارد؟
نویز ضربه‌ای تعداد محدودی نمونه را به مقادیر حدی می‌برد و خطاهای بزرگ و پراکنده ایجاد می‌کند. نویز گاوسی خطایی جمع‌شونده با دامنه‌های متفاوت در سراسر تصویر ایجاد می‌کند؛ برش شدت‌ها در نزدیکی 0 و 255 توزیع نویز مشاهده‌شده را تغییر می‌دهد. این تفاوت، انتخاب میان فیلتر مقاوم به نقاط پرت و میانگین‌گیری را تعیین می‌کند.

</div>

In [ ]:
document_path = Path("Media_src/scanned_document.png")
document_color = cv2.imread(str(document_path), cv2.IMREAD_COLOR)
if document_color is None:
    raise FileNotFoundError(f"Could not load image: {document_path}")

document = cv2.cvtColor(document_color, cv2.COLOR_BGR2GRAY)
print(
    f"Original image shape: {document_color.shape}, "
    f"dtype: {document_color.dtype}, channel order: BGR"
)
print(
    f"Grayscale shape: {document.shape}, dtype: {document.dtype}, "
    f"range: {document.min()} to {document.max()}"
)

rng = np.random.default_rng(42)

# noise_density = 0.2

noise_density = 0.05
random_values = rng.random(document.shape)
salt_pepper = document.copy()

salt_pepper[random_values < noise_density / 2] = 0
salt_pepper[
    (random_values >= noise_density / 2) & (random_values < noise_density)
] = 255

# gaussian_sigma = 150

gaussian_sigma = 25
gaussian_noise = rng.normal(0, gaussian_sigma, document.shape)
gaussian_noisy = np.clip(
    document.astype(np.float32) + gaussian_noise, 0, 255
).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, picture, name in zip(
    axes,
    [document, salt_pepper, gaussian_noisy],
    ["Original", f"Salt and pepper ({noise_density * 100:.0f}%)", f"Gaussian (sigma = {gaussian_sigma})"],
):
    ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۲. مفهوم کرنل و پیاده‌سازی فیلترگذاری

**صورت مسئله و خواسته تمرین:** مفهوم کرنل توضیح داده و فیلترگذاری مکانی دستی برای کرنل‌هایی با ابعاد مختلف، همراه با تعیین رفتار مرزی، پیاده‌سازی شود.

**روش کار و منطق ریاضی:** کانولوشن مکانی (Spatial Convolution) به صورت $Y(x,y)=\sum_{i,j}K(i,j)I(x-i,y-j)$ است. کد، کرنل را در هر دو محور برمی‌گرداند و حاصل‌ضرب برش‌های همسایگی را جمع می‌کند. ابعاد کرنل دوبعدی باید فرد باشند و محدود به 3×3 نیستند. امتداد مرزی با `reflect`، بدون تکرار پیکسل مرزی، انجام و خروجی به صورت `float32` نگه داشته می‌شود.

**شواهد عددی و نتایج تجربی:** این بخش خروجی عددی مستقلی چاپ نمی‌کند؛ مقایسهٔ کمی پیاده‌سازی با توابع آماده در قسمت ۵ گزارش شده است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### کرنل چه کاربردی دارد و چرا رفتار مرزی مهم است؟
ضرایب کرنل تعیین می‌کنند کدام تغییرات مکانی تقویت یا تضعیف شوند؛ برای نمونه هموارسازی و استخراج مشتق از همین چارچوب بهره می‌برند. در نزدیکی لبهٔ تصویر، بخشی از همسایگی وجود ندارد؛ انعکاس، ادامه‌ای مصنوعی با پرش کمتر نسبت به پرکردن با صفر می‌سازد، اما همچنان فرضی دربارهٔ دادهٔ خارج از تصویر است. خروجی علامت‌دار برای حفظ پاسخ مثبت و منفی مشتق ضروری است.

</div>

In [ ]:
def spatial_convolution(
    gray_image: np.ndarray, kernel: np.ndarray
) -> np.ndarray:
    kernel = np.asarray(kernel, dtype=np.float32)
    if (
        gray_image.ndim != 2
        or kernel.ndim != 2
        or any(size % 2 == 0 for size in kernel.shape)
    ):
        raise ValueError(
            "The image must be grayscale and the kernel "
            "must have odd dimensions."
        )
    pad_y, pad_x = kernel.shape[0] // 2, kernel.shape[1] // 2
    padded = np.pad(
        gray_image.astype(np.float32),
        ((pad_y, pad_y), (pad_x, pad_x)),
        mode="reflect",
    )
    result = np.zeros(gray_image.shape, dtype=np.float32)
    flipped_kernel = kernel[::-1, ::-1]
    height, width = gray_image.shape
    for row in range(kernel.shape[0]):
        for col in range(kernel.shape[1]):
            result += (
                flipped_kernel[row, col]
                * padded[row : row + height, col : col + width]
            )
    return result


def to_uint8(values: np.ndarray) -> np.ndarray:
    return np.clip(np.rint(values), 0, 255).astype(np.uint8)

<div dir="rtl">

### ۳. اعمال و مقایسه فیلترهای پایین‌گذر

**صورت مسئله و خواسته تمرین:** فیلترهای دستی Mean با کرنل 3×3، Gaussian با کرنل 5×5 و Median با پنجرهٔ 3×3 روی هر دو تصویر نویزدار اعمال و از نظر کاهش نویز و حفظ جزئیات مقایسه شوند.

**روش کار و منطق ریاضی:** کرنل میانگین شامل 9 ضریب $1/9$ است. کرنل گاوسی $K=ww^T/256$ با $w=[1,4,6,4,1]^T$ است. میانه مقدار میانی 9 نمونهٔ پنجره را انتخاب می‌کند و فیلتر خطی کانولوشنی نیست. مرز میانه با تکرار پیکسل و مرز دو فیلتر خطی با انعکاس مدیریت می‌شود؛ برای خروجی تصویری پاسخ خطی گرد و به 0 تا 255 بریده می‌شود.

**شواهد عددی و نتایج تجربی:** برای هر نویز، تصویر نویزدار و سه نتیجه در شکل نمایش داده شده‌اند. PSNR دستی فلفل‌نمکی برای Mean، Gaussian و Median به‌ترتیب 21.24، 22.74 و 36.39 دسی‌بل؛ برای نویز گاوسی 26.38، 26.64 و 28.53 دسی‌بل است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### کدام فیلتر برای نویز فلفل‌نمکی مناسب‌تر است و چرا؟
میانه در این آزمایش برتر است. تا زمانی که اکثریت پنجره آلوده نباشد، نمونهٔ حدی معمولاً میانه نمی‌شود؛ در مقابل میانگین و گاوسی مقدار پرت را در همسایگی پخش می‌کنند و لکه‌هایی با شدت متوسط می‌سازند.

#### رفتار فیلترها در برابر نویز گاوسی چگونه است؟
میانگین‌گیری و گاوسی نوسانات مستقل را با ترکیب نمونه‌ها کاهش می‌دهند؛ وزن مرکزی بیشتر گاوسی نسبت به پنجرهٔ یکنواخت بر رفتار محلی اثر می‌گذارد. در این تصویر میانه بالاترین PSNR را دارد، اما این نتیجه به محتوای سند و پارامترها وابسته است و برتری عمومی آن برای هر نویز گاوسی را نشان نمی‌دهد.

#### آیا کاهش نویز همواره جزئیات را حفظ می‌کند؟
خیر. هموارسازی تغییرات شدید متن و پس‌زمینه را نیز کاهش می‌دهد و مرز حروف را پخش می‌کند. میانه نیز ممکن است خطوط باریک یا اجزای کوچک را که در پنجره در اقلیت‌اند حذف کند.

</div>

In [ ]:
mean_kernel = np.ones((3, 3), dtype=np.float32) / 9
gaussian_weights = np.array([1, 4, 6, 4, 1], dtype=np.float32)
gaussian_kernel = np.outer(gaussian_weights, gaussian_weights) / 256


def median_filter(gray_image: np.ndarray, window_size: int = 3) -> np.ndarray:
    if gray_image.ndim != 2 or window_size < 1 or window_size % 2 == 0:
        raise ValueError(
            "The image must be grayscale and the window size "
            "must be positive and odd."
        )
    radius = window_size // 2
    padded = np.pad(gray_image, radius, mode="edge")
    height, width = gray_image.shape
    neighbors = np.stack(
        [
            padded[row : row + height, col : col + width]
            for row in range(window_size)
            for col in range(window_size)
        ]
    )
    return np.partition(neighbors, window_size * window_size // 2, axis=0)[
        window_size * window_size // 2
    ]


noisy_images = {"Salt and pepper": salt_pepper, "Gaussian": gaussian_noisy}
manual_results = {}
for noise_name, noisy_image in noisy_images.items():
    manual_results[noise_name] = {
        "Mean": to_uint8(spatial_convolution(noisy_image, mean_kernel)),
        "Gaussian": to_uint8(
            spatial_convolution(noisy_image, gaussian_kernel)
        ),
        "Median": median_filter(noisy_image, 3),
    }

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for row, (noise_name, noisy_image) in enumerate(noisy_images.items()):
    pictures = [noisy_image] + list(manual_results[noise_name].values())
    for ax, picture, title in zip(
        axes[row],
        pictures,
        [noise_name, "Mean 3x3", "Gaussian 5x5", "Median 3x3"],
    ):
        ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۴. اعمال فیلترهای بالاگذر و استخراج لبه‌ها

**صورت مسئله و خواسته تمرین:** Sobel و Laplacian دستی بر سند اصلی، سند با نویز فلفل‌نمکی و نسخهٔ بازیابی‌شده با میانه اعمال و تأثیر نویز بر لبه‌ها تحلیل شود.

**روش کار و منطق ریاضی:** Sobel دو پاسخ جهت‌دار $G_x,G_y$ تولید می‌کند و برای نمایش اندازهٔ گرادیان $G=\sqrt{G_x^2+G_y^2}$ به کار می‌رود. Laplacian با کرنل چهارهمسایگی $[[0,1,0],[1,-4,1],[0,1,0]]$ تغییرات مرتبهٔ دوم را تقریب می‌زند. پاسخ‌های خام اعشاری‌اند؛ برای نمایش، اندازهٔ Sobel و قدرمطلق Laplacian به بازهٔ 0 تا 255 محدود می‌شوند.

**شواهد عددی و نتایج تجربی:** شکل سه ردیف متناظر با تصویر اصلی، نویزدار و بازیابی‌شده و سه ستون ورودی، Sobel و Laplacian دارد. مقایسهٔ پاسخ‌های خام با OpenCV در قسمت بعدی آمده است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### نویز چگونه تشخیص لبه را تغییر می‌دهد؟
نقاط ضربه‌ای تغییر شدت ناگهانی ایجاد می‌کنند و هر دو فیلتر به آن‌ها پاسخ می‌دهند. لاپلاسین (Laplacian) تقریب مشتق مرتبهٔ دوم شدت است: به تغییرات شدید شیب حساس است، بنابراین نوسان سریع و ناگهانی ناشی از نویز را برجسته می‌کند و پاسخ‌هایی مشابه لبه‌های واقعی، یعنی لبه‌های کاذب (False Edges)، به وجود می‌آورد. Sobel مشتق مرتبهٔ اول را همراه با هموارسازی محدود در راستای عمود بر مشتق محاسبه می‌کند، اما آن نیز به نویز ضربه‌ای حساس است؛ میزان حساسیت نسبی به کرنل، مقیاس و شدت نویز بستگی دارد.

#### چرا اعمال Median پیش از استخراج لبه نتیجه را تغییر می‌دهد؟
میانه نقاط حدی منزوی را کاهش می‌دهد و در نتیجه پاسخ‌های کاذب کمتری به فیلتر لبه می‌رسد. با این حال، اگر جزئیات ظریف حروف در اکثریت پنجره نباشند، میانه ممکن است آن‌ها را تغییر دهد و بخشی از لبه‌های واقعی نیز تضعیف شوند.

</div>

In [ ]:
sobel_x_kernel = np.array(
    [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32
)
sobel_y_kernel = sobel_x_kernel.T
laplacian_kernel = np.array(
    [[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32
)


def manual_edges(
    gray_image: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    gx = spatial_convolution(gray_image, sobel_x_kernel)
    gy = spatial_convolution(gray_image, sobel_y_kernel)
    laplacian = spatial_convolution(gray_image, laplacian_kernel)
    return gx, gy, laplacian


edge_inputs = {
    "Original": document,
    "Salt and pepper": salt_pepper,
    "Median restored": manual_results["Salt and pepper"]["Median"],
}
manual_edge_results = {
    name: manual_edges(picture) for name, picture in edge_inputs.items()
}
fig, axes = plt.subplots(3, 3, figsize=(12, 13))
for row, (name, picture) in enumerate(edge_inputs.items()):
    gx, gy, laplacian = manual_edge_results[name]
    displays = [
        picture,
        to_uint8(np.hypot(gx, gy)),
        to_uint8(np.abs(laplacian)),
    ]
    for ax, result, title in zip(
        axes[row], displays, [name, "Sobel magnitude", "Laplacian absolute"]
    ):
        ax.imshow(result, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۵. مقایسه با توابع آماده و تحلیل نتایج

**صورت مسئله و خواسته تمرین:** فیلترهای دستی با پیاده‌سازی OpenCV مقایسه، PSNR تصاویر بازیابی‌شده محاسبه و رابطهٔ کاهش نویز با حفظ جزئیات بررسی شود.

**روش کار و منطق ریاضی:** برای فیلترهای خطی از مرز `BORDER_REFLECT_101` و برای میانه از تکرار مرز استفاده شده است. چون Sobel آماده همبستگی و تابع دستی کانولوشن انجام می‌دهد، علامت Sobel آماده پیش از مقایسه معکوس می‌شود. با $MSE=N^{-1}\sum_i(I_i-\hat I_i)^2$، معیار $PSNR=10\log_{10}(255^2/MSE)$ بر حسب دسی‌بل محاسبه می‌شود. اختلاف فیلترها روی داده‌های خام یا خروجی ۸ بیتی متناظر محاسبه شده است.

**شواهد عددی و نتایج تجربی:**

| نویز | فیلتر | PSNR دستی (dB) | PSNR آماده (dB) | بیشینه اختلاف | میانگین اختلاف |
|---|---|---:|---:|---:|---:|
| فلفل‌نمکی | Mean | 23.78 | 23.78 | 0 | 0.0000 |
| فلفل‌نمکی | Gaussian | 24.73 | 24.75 | 1 | 0.0272 |
| فلفل‌نمکی | Median | 36.39 | 36.39 | 0 | 0.0000 |
| گاوسی | Mean | 26.38 | 26.38 | 0 | 0.0000 |
| گاوسی | Gaussian | 26.64 | 26.64 | 1 | 0.0020 |
| گاوسی | Median | 28.53 | 28.53 | 0 | 0.0000 |

PSNR ورودی نویزدار فلفل‌نمکی 16.35 و گاوسی 22.55 دسی‌بل است. بیشینهٔ اختلاف پاسخ خام هر دو مؤلفهٔ Sobel و Laplacian برای هر سه ورودی آزمایش‌شده صفر است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### اختلاف دستی و OpenCV از کجا می‌آید؟
میانگین و میانه در این داده‌ها یکسان‌اند. اختلاف حداکثر یک واحدی گاوسی با تفاوت مسیر محاسبات عددی و گرد کردن سازگار است؛ همسان‌سازی نوع داده و رفتار مرزی برای مقایسهٔ معتبر ضروری است. با تطبیق علامت، پاسخ‌های خام لبه در این آزمایش برابرند.

#### کدام فیلتر برای هر نویز مناسب‌تر است؟
میانه برای هر دو نویز، در این آزمایش، بیشترین PSNR را دارد؛ برتری برای نویز ضربه‌ای با مقاومت آن به نقاط پرت سازگار است. برای نویز گاوسی نیز ساختار عمدتاً یکنواخت سند و حاشیه‌های تیز می‌توانند به نتیجهٔ میانه کمک کنند، اما برای داده‌های دیگر این رتبه‌بندی تضمین‌شده نیست.

#### آیا بهترین PSNR به معنی حفظ بهتر همهٔ حروف است؟
خیر. PSNR خطای پیکسلی را بر کل تصویر میانگین می‌گیرد؛ پس تغییر یک خط باریک ممکن است در مقابل مساحت بزرگ پس‌زمینه تأثیر کمی داشته باشد. برش حروف در خروجی بعدی برای بررسی بصری مکمل ارائه شده است، ولی به‌تنهایی اندازه‌گیری عینی خوانایی نیست.

</div>

In [ ]:
opencv_results = {}
for noise_name, noisy_image in noisy_images.items():
    opencv_results[noise_name] = {
        "Mean": cv2.blur(
            noisy_image, (3, 3), borderType=cv2.BORDER_REFLECT_101
        ),
        "Gaussian": cv2.GaussianBlur(
            noisy_image, (5, 5), 0, borderType=cv2.BORDER_REFLECT_101
        ),
        "Median": cv2.medianBlur(noisy_image, 3),
    }


def psnr(
    reference: np.ndarray, result: np.ndarray
) -> Union[float, np.float64]:
    difference = reference.astype(np.float64) - result.astype(np.float64)
    mse = np.mean(difference**2)
    return float("inf") if mse == 0 else 10 * np.log10(255**2 / mse)


print(
    f"{'Noise':<17} {'Filter':<10} {'Manual PSNR':>12} "
    f"{'OpenCV PSNR':>12} {'Max diff':>9} {'Mean diff':>10}"
)
for noise_name in noisy_images:
    for filter_name in manual_results[noise_name]:
        manual = manual_results[noise_name][filter_name]
        ready = opencv_results[noise_name][filter_name]
        difference = np.abs(manual.astype(np.int16) - ready.astype(np.int16))
        print(
            f"{noise_name:<17} {filter_name:<10} "
            f"{psnr(document, manual):>10.2f} dB "
            f"{psnr(document, ready):>10.2f} dB "
            f"{difference.max():>9} {difference.mean():>10.4f}"
        )
    print(
        f"{noise_name} noisy input PSNR: "
        f"{psnr(document, noisy_images[noise_name]):.2f} dB"
    )

print(
    "\nMaximum absolute difference between raw manual "
    "and OpenCV edge responses:"
)
for name, picture in edge_inputs.items():
    gx, gy, laplacian = manual_edge_results[name]
    cv_gx = -cv2.Sobel(
        picture, cv2.CV_32F, 1, 0, ksize=3, borderType=cv2.BORDER_REFLECT_101
    )
    cv_gy = -cv2.Sobel(
        picture, cv2.CV_32F, 0, 1, ksize=3, borderType=cv2.BORDER_REFLECT_101
    )
    cv_laplacian = cv2.Laplacian(
        picture, cv2.CV_32F, ksize=1, borderType=cv2.BORDER_REFLECT_101
    )
    print(
        f"{name}: Sobel X = {np.max(np.abs(gx - cv_gx)):.0f}, "
        f"Sobel Y = {np.max(np.abs(gy - cv_gy)):.0f}, "
        f"Laplacian = {np.max(np.abs(laplacian - cv_laplacian)):.0f}"
    )

In [ ]:
crop = np.s_[930:1230, 150:900]
fig, axes = plt.subplots(2, 5, figsize=(20, 6))
for row, (noise_name, noisy_image) in enumerate(noisy_images.items()):
    pictures = [document, noisy_image] + list(
        manual_results[noise_name].values()
    )
    for ax, picture, title in zip(
        axes[row],
        pictures,
        ["Original", noise_name, "Mean", "Gaussian", "Median"],
    ):
        ax.imshow(picture[crop], cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

## بخش سوم: پردازش و بازیابی ویدئو

</div>

<div dir="rtl">

### ۱. بارگذاری و بررسی مشخصات ویدئو

**صورت مسئله و خواسته تمرین:** تعداد فریم، نرخ فریم، ابعاد و مدت ویدئوی ورودی گزارش شود.

**روش کار و منطق ریاضی:** ویژگی‌ها با `VideoCapture` خوانده می‌شوند و مدت تخمینی از رابطهٔ $T=N/f$ به دست می‌آید؛ این محاسبه نرخ فریم ثابت گزارش‌شده را فرض می‌کند.

**شواهد عددی و نتایج تجربی:**

| مشخصه | مقدار |
|---|---:|
| تعداد فریم | 450 |
| نرخ فریم | 30.00 fps |
| عرض × ارتفاع | 1280 × 720 |
| مدت محاسبه‌شده | 15.00 ثانیه |

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### چگونه مدت ویدئو تعیین شد؟
با تقسیم 450 فریم بر 30 فریم‌برثانیه، مدت 15 ثانیه به دست می‌آید. این مقدار حاصل مشخصات ثبت‌شدهٔ ویدئو است و ارزیابی جداگانهٔ زمان‌بندی تک‌فریم‌ها انجام نشده است.

</div>

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

video_path = Path("Media_src/park_video.mp4")
capture = cv2.VideoCapture(str(video_path))
if not capture.isOpened():
    raise FileNotFoundError(f"Could not open video: {video_path}")
frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
fps = capture.get(cv2.CAP_PROP_FPS)
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
capture.release()
if frame_count <= 0 or fps <= 0 or width <= 0 or height <= 0:
    raise ValueError("Video properties are invalid.")
print(f"Frame count: {frame_count}")
print(f"FPS: {fps:.2f}")
print(f"Frame dimensions (width x height): {width} x {height}")
print(f"Duration: {frame_count / fps:.2f} seconds")

<div dir="rtl">

### ۲. اضافه کردن نویز به ویدئو

**صورت مسئله و خواسته تمرین:** نویز فلفل‌نمکی با چگالی 5٪ و تعداد تقریباً برابر نقاط نمکی و فلفلی به تمام فریم‌ها افزوده و ویدئوی `noisy_video.mp4` و یک مقایسهٔ تصویری ایجاد شوند.

**روش کار و منطق ریاضی:** برای هر فریم $N=1280\times720$ موقعیت، $0.05N=46080$ موقعیت متمایز بدون جایگذاری انتخاب می‌شود؛ نیمهٔ اول هر سه کانال را صفر و نیمهٔ دوم هر سه را 255 می‌کند. بذر تصادفی 42 است. ویدئو با `mp4v` و ویژگی‌های ورودی در `Media_out` نوشته می‌شود.

**شواهد عددی و نتایج تجربی:** در هر فریم 46080 موقعیت، شامل 23040 نقطه از هر نوع، پیش از فشرده‌سازی انتخاب می‌شود؛ فایل خروجی 450 فریم دارد و فریم اول اصلی و نویزدار نمایش داده شده‌اند.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### آیا فایل ذخیره‌شده دقیقاً ۵٪ پیکسل صفر یا ۲۵۵ دارد؟
تعداد موقعیت‌های انتخاب‌شده پیش از رمزگذاری دقیقاً 5٪ است، ولی برخی مقادیر ممکن است از قبل حدی باشند. به‌علاوه فشرده‌سازی اتلافی مقادیر را تغییر می‌دهد و تضمینی برای باقی‌ماندن دقیق مقادیر 0 و 255 در ویدئوی بازخوانی‌شده وجود ندارد.

</div>

In [ ]:
output_dir = Path("Media_out")
output_dir.mkdir(exist_ok=True)
noisy_path = output_dir / "noisy_video.mp4"
median_path = output_dir / "filtered_video_1.mp4"
gaussian_path = output_dir / "filtered_video_2.mp4"
video_codec = cv2.VideoWriter_fourcc(*"mp4v")


def open_video_writer(path: Union[str, Path]) -> cv2.VideoWriter:
    writer = cv2.VideoWriter(str(path), video_codec, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not create video: {path}")
    return writer


rng_video = np.random.default_rng(42)
noisy_capture = cv2.VideoCapture(str(video_path))
noisy_writer = open_video_writer(noisy_path)
processed_frames = 0
noise_pixels = int(round(height * width * 0.05))
try:
    while True:
        success, frame = noisy_capture.read()
        if not success:
            break
        noisy_frame = frame.copy()
        positions = rng_video.choice(
            height * width, size=noise_pixels, replace=False
        )
        pixels = noisy_frame.reshape(-1, 3)
        pixels[positions[: noise_pixels // 2]] = 0
        pixels[positions[noise_pixels // 2 :]] = 255
        noisy_writer.write(noisy_frame)
        processed_frames += 1
finally:
    noisy_capture.release()
    noisy_writer.release()
if processed_frames != frame_count:
    raise RuntimeError(
        f"Read {processed_frames} frames but metadata says {frame_count}."
    )
print(f"Noisy video saved: {noisy_path} ({processed_frames} frames)")
print(
    f"Changed positions per frame: {noise_pixels} "
    f"({noise_pixels / (height * width) * 100:.2f}%)"
)

In [ ]:
def read_frame(path: Union[str, Path], index: int) -> np.ndarray:
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {path}")
    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, index)
        success, frame = capture.read()
        if not success:
            raise RuntimeError(f"Could not read frame {index} from {path}")
        return frame
    finally:
        capture.release()


first_original = read_frame(video_path, 0)
first_noisy = read_frame(noisy_path, 0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, frame, name in zip(
    axes,
    [first_original, first_noisy],
    ["Original: frame 0", "Noisy: frame 0"],
):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۳. حذف نویز از ویدئو

**صورت مسئله و خواسته تمرین:** یک فیلتر مناسب برای تمام فریم‌های نویزدار انتخاب و نتیجه با نام `filtered_video_1.mp4` ذخیره شود؛ نام، پارامتر و دلیل انتخاب ارائه شوند.

**روش کار و منطق ریاضی:** فیلتر میانه (Median Filter) با پنجرهٔ 3×3 و تابع `medianBlur` به هر فریم خوانده‌شده از ویدئوی نویزدار اعمال می‌شود. هر مؤلفهٔ رنگی با میانهٔ همسایگی متناظر جایگزین می‌شود. خروجی با همان نرخ و ابعاد در `Media_out` ذخیره می‌شود.

**شواهد عددی و نتایج تجربی:** خروجی میانه شامل 450 فریم است. میانگین PSNR سه فریم نمونهٔ فایل ذخیره‌شده نسبت به فریم‌های اصلی 24.45 دسی‌بل است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### چرا فیلتر میانه انتخاب شد؟
نویز فلفل‌نمکی نمونه‌های پرت با دامنهٔ زیاد ایجاد می‌کند. میانه بر خلاف میانگین، این مقادیر را مستقیماً در حاصل جمع نمی‌کند و در همسایگی‌هایی که اکثریت آن سالم‌اند، معمولاً ضربه‌های منفرد را حذف می‌کند. با وجود این، بافت ظریف و خطوط باریک نیز ممکن است در پنجره تغییر کنند.

</div>

<div dir="rtl">

### ۴. مقایسه روش‌های حذف نویز

**صورت مسئله و خواسته تمرین:** فیلتر دوم با عملکرد متفاوت به‌طور مستقل روی همان ویدئوی نویزدار اعمال و نتیجه با فیلتر اول از نظر حذف نویز، حفظ جزئیات و تاری مقایسه شود.

**روش کار و منطق ریاضی:** فیلتر دوم Gaussian خطی با کرنل 5×5 و پارامتر `sigma=0` در OpenCV است؛ مقدار صفر از OpenCV می‌خواهد انحراف معیار را بر اساس اندازهٔ کرنل تعیین کند. خروجی آن `filtered_video_2.mp4` است. هر دو روش از فریم‌های یکسان فایل نویزدار استفاده می‌کنند و زنجیره‌ای نیستند.

**شواهد عددی و نتایج تجربی:** میانگین PSNR سه فریم، پس از ذخیره و بازخوانی، برای میانه 24.45 و گاوسی 22.26 دسی‌بل است؛ فایل هر دو خروجی 450 فریم دارد.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### کدام روش نویز فلفل‌نمکی را بهتر حذف کرده است؟
در سه فریم مقایسه‌شده، میانه PSNR بالاتری دارد و نقاط ضربه‌ای برجسته را بهتر کاهش می‌دهد. گاوسی مقدار نقاط پرت را با همسایگی ترکیب می‌کند و می‌تواند اثر آن‌ها را در فضای اطراف پخش کند.

#### کدام روش لبه‌ها را بهتر حفظ می‌کند و چرا تصویر تار می‌شود؟
در مقایسهٔ تصویری، میانه مرزهای مسیر و ساختارهای درشت را نسبت به گاوسی بهتر حفظ می‌کند، هرچند جزئیات ظریف هر دو خروجی نرم‌تر می‌شوند. گاوسی یک پایین‌گذر است و برای کاهش تغییرات سریع ناشی از نویز، تغییرات سریع ناشی از لبهٔ واقعی را نیز تضعیف می‌کند؛ بنابراین تاری پیامد مبادلهٔ کاهش نویز و حفظ جزئیات است.

</div>

In [ ]:
filter_capture = cv2.VideoCapture(str(noisy_path))
if not filter_capture.isOpened():
    raise RuntimeError(f"Could not open noisy video: {noisy_path}")
median_writer = open_video_writer(median_path)
try:
    gaussian_writer = open_video_writer(gaussian_path)
except Exception:
    median_writer.release()
    filter_capture.release()
    raise
filtered_frames = 0
try:
    while True:
        success, noisy_frame = filter_capture.read()
        if not success:
            break
        median_frame = cv2.medianBlur(noisy_frame, 3)
        gaussian_frame = cv2.GaussianBlur(noisy_frame, (5, 5), 0)
        median_writer.write(median_frame)
        gaussian_writer.write(gaussian_frame)
        filtered_frames += 1
finally:
    filter_capture.release()
    median_writer.release()
    gaussian_writer.release()
if filtered_frames != processed_frames:
    raise RuntimeError(
        f"Filtered {filtered_frames} of {processed_frames} noisy frames."
    )
print(f"Median output: {median_path}, {filtered_frames} frames")
print(f"Gaussian output: {gaussian_path}, {filtered_frames} frames")

<div dir="rtl">

### ۵. بررسی و تحلیل نتایج

**صورت مسئله و خواسته تمرین:** سه فریم ابتدایی، میانی و انتهایی در چهار حالت مقایسه شوند؛ تأثیر بر جزئیات محیط، لبه‌ها و اجسام متحرک و حفظ ابعاد و نرخ فریم بررسی شود.

**روش کار و منطق ریاضی:** فریم‌های با اندیس صفرمبنای 0، 225 و 449 از فایل‌های اصلی، نویزدار و دو خروجی بازخوانی می‌شوند. PSNR هر فریم بر تمام مؤلفه‌های رنگی با $10\log_{10}(255^2/MSE)$ محاسبه می‌شود و میانگین سه مقدار به عنوان خلاصه گزارش می‌شود. کد ویژگی‌های فایل‌های بازشده را نیز با ورودی مقایسه می‌کند.

**شواهد عددی و نتایج تجربی:**

| فریم | نویزدار (dB) | میانه (dB) | گاوسی (dB) |
|---|---:|---:|---:|
| 0 | 17.52 | 23.71 | 21.40 |
| 225 | 18.47 | 24.73 | 22.62 |
| 449 | 18.49 | 24.92 | 22.78 |
| میانگین سه فریم | 18.16 | 24.45 | 22.26 |

هر چهار فایل در مشخصات خوانده‌شده 450 فریم، 30.00 فریم‌برثانیه و ابعاد 1280×720 دارند. مقایسه شامل اثر فشرده‌سازی فایل‌های خروجی است.

**پاسخ به پرسش‌های تحلیلی پروژه:**

#### فیلترها بر جزئیات محیط و لبه‌ها چه اثری دارند؟
در فریم‌ها، بافت برگ‌ها و مرز مسیر در خروجی‌ها نسبت به اصل نرم‌ترند؛ میانگین‌گیری گاوسی باعث هموارشدن بیشتر بافت و مرزها می‌شود. میانه در این سه نمونه PSNR بالاتر و حذف مؤثرتر نقاط برجسته دارد، اما معیار پیکسلی به‌تنهایی کیفیت بافت را کامل توصیف نمی‌کند.

#### دربارهٔ اجسام متحرک چه می‌توان نتیجه گرفت؟
هر فریم مستقل و بدون فیلترگذاری زمانی (Temporal Filtering) یا هم‌ترازی حرکت پردازش می‌شود. فیلتر گاوسی مکانی می‌تواند لبه‌ها و جزئیات جسم متحرک را در هر فریم نرم‌تر کند و تاری موجود ناشی از حرکت (Motion Blur) را از نظر ظاهری تشدید کند؛ با این حال، خود این فیلتر فریم‌های زمانی را با هم ترکیب نمی‌کند و بنابراین به‌تنهایی تاری حرکتی ناشی از ترکیب زمانی ایجاد نمی‌کند. در فیلتر میانه نیز تغییر نویز و محتوای پنجره از فریمی به فریم دیگر می‌تواند جابه‌جایی ظاهری مرز (Edge Jittering) یا سوسوزدن (Flickering) ایجاد کند. وقوع و شدت این پدیده‌ها از سه تصویر ثابت قابل اثبات نیست و به مشاهدهٔ ویدئو یا سنجش سازگاری زمانی نیاز دارد.

#### آیا نرخ و ابعاد در خروجی‌ها حفظ شده‌اند؟
بله؛ جدول مشخصات ذخیره‌شده نشان می‌دهد تعداد فریم، نرخ و ابعاد هر سه خروجی با ورودی برابر است. این بررسی از فرادادهٔ فایل‌ها استفاده می‌کند و آزمون جداگانهٔ زمان‌بندی تک‌فریم‌ها نیست.

</div>

In [ ]:
def frame_psnr(
    reference: np.ndarray, result: np.ndarray
) -> Union[float, np.float64]:
    difference = reference.astype(np.float32) - result.astype(np.float32)
    mse = np.mean(difference**2)
    return float("inf") if mse == 0 else 10 * np.log10(255**2 / mse)


video_paths = [video_path, noisy_path, median_path, gaussian_path]
video_names = ["Original", "Noisy", "Median 3x3", "Gaussian 5x5"]
sample_indices = [0, frame_count // 2, frame_count - 1]
sample_scores = {name: [] for name in video_names[1:]}
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for row, index in enumerate(sample_indices):
    frames = [read_frame(path, index) for path in video_paths]
    for col, (ax, frame, name) in enumerate(
        zip(axes[row], frames, video_names)
    ):
        title = f"{name} | frame {index}"
        if col > 0:
            score = frame_psnr(frames[0], frame)
            sample_scores[name].append(score)
            title += f"\nPSNR: {score:.2f} dB"
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()
print("Mean PSNR over the three selected frames:")
for name, scores in sample_scores.items():
    print(f"{name}: {np.mean(scores):.2f} dB")

In [ ]:
print(
    f"{'Video':<16} {'Frames':>8} {'FPS':>8} "
    f"{'Width':>8} {'Height':>8} {'Matches input':>15}"
)
for path, name in zip(video_paths, video_names):
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {path}")
    count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    rate = capture.get(cv2.CAP_PROP_FPS)
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()
    matches = (
        count == frame_count
        and abs(rate - fps) < 0.01
        and (frame_width, frame_height) == (width, height)
    )
    print(
        f"{name:<16} {count:>8} {rate:>8.2f} "
        f"{frame_width:>8} {frame_height:>8} {str(matches):>15}"
    )